> Resetto le variabili.

In [ ]:
%reset

> Importo moduli.



In [ ]:
import os
import glob
import nibabel
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


# PRE-PROCESSING

> Variabili utili.

In [ ]:
image_size = np.array([61,73,61])

> Creo matrice dei pazienti.

In [ ]:
os.chdir('/content/drive/My Drive/Brain - Tatiana/Immagini input/VMHC/1') 
images_pazienti = glob.glob('*.nii', recursive=True)
num_pazienti = len(images_pazienti) 

X_pazienti = np.zeros((num_pazienti,np.product(image_size)))
t = 0
for fMRI in images_pazienti:
  file_nii = nibabel.load(fMRI)
  img = np.array(file_nii.dataobj)
  X_pazienti[t,:] = np.reshape(img,(1,np.product(image_size)))
  t = t + 1

y_pazienti = np.ones((num_pazienti,1))

> Creao matrice dei controlli.

In [ ]:
os.chdir('/content/drive/My Drive/Brain - Tatiana/Immagini input/VMHC/3') 
images_controlli = glob.glob('*.nii', recursive=True) 
num_controlli = len(images_controlli)

X_controlli = np.zeros((num_controlli,np.product(image_size)))
t = 0
for fMRI in images_controlli:
  file_nii = nibabel.load(fMRI)
  img = np.array(file_nii.dataobj)
  X_controlli[t,:] = np.reshape(img,(1,np.product(image_size)))
  t = t + 1

y_controlli = np.zeros((num_controlli,1))

> Unisco le matrici.

In [ ]:
X = np.concatenate((X_pazienti,X_controlli))
y = np.concatenate((y_pazienti,y_controlli))

Elimino dalla matrice X le colonne con somma 0 (i.e. le colonne corrispondenti a voxel neri in tutti i soggetti)

In [ ]:
initial_num_cols = X.shape[1]

mask = (X == 0).all(0)
column_indices = np.where(mask)[0]
X = X[:,~mask]

final_num_cols = X.shape[1]

print(str(initial_num_cols - final_num_cols) + ' columns were dropped from the dataset')


198596 columns were dropped from the dataset


# PCA

In [ ]:
scaler1 = StandardScaler()
scaler1.fit(X)
X_scaled = scaler1.transform(X)
X_pca = X_scaled

pca = PCA(n_components=25)
pca.fit(X_scaled)
X_pca = pca.transform(X_scaled)
#


In [ ]:
X.shape

(30, 73037)

# Parameter estimation

In [16]:
# Set the parameters by cross-validation
tuned_parameters = [{'kernel': ['rbf','linear','sigmoid','poly'], 'gamma': [1e-2,1e-1,1e-3, 1e-4, 'scale', 'auto'],
                     'C': [1, 10, 100, 1000]}]

clf = GridSearchCV(svm.SVC(), tuned_parameters, scoring='accuracy', cv=5)
clf.fit(X_pca, y.ravel())

print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
print("Grid scores on development set:")
print()
means = clf.cv_results_['mean_test_score']
stds = clf.cv_results_['std_test_score']
for mean, std, params in zip(means, stds, clf.cv_results_['params']):
        print("%0.3f (+/-%0.03f) for %r"
              % (mean, std * 2, params))

Best parameters set found on development set:

{'C': 1, 'gamma': 0.01, 'kernel': 'linear'}

Grid scores on development set:

0.600 (+/-0.400) for {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
0.867 (+/-0.249) for {'C': 1, 'gamma': 0.01, 'kernel': 'linear'}
0.667 (+/-0.558) for {'C': 1, 'gamma': 0.01, 'kernel': 'sigmoid'}
0.567 (+/-0.163) for {'C': 1, 'gamma': 0.01, 'kernel': 'poly'}
0.467 (+/-0.133) for {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
0.867 (+/-0.249) for {'C': 1, 'gamma': 0.1, 'kernel': 'linear'}
0.600 (+/-0.452) for {'C': 1, 'gamma': 0.1, 'kernel': 'sigmoid'}
0.567 (+/-0.163) for {'C': 1, 'gamma': 0.1, 'kernel': 'poly'}
0.467 (+/-0.133) for {'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}
0.867 (+/-0.249) for {'C': 1, 'gamma': 0.001, 'kernel': 'linear'}
0.767 (+/-0.340) for {'C': 1, 'gamma': 0.001, 'kernel': 'sigmoid'}
0.567 (+/-0.163) for {'C': 1, 'gamma': 0.001, 'kernel': 'poly'}
0.500 (+/-0.000) for {'C': 1, 'gamma': 0.0001, 'kernel': 'rbf'}
0.867 (+/-0.249) for {'C': 1, 'gamma': 0.00

# Leave-one out

In [ ]:
# evaluate model
best_clf = clf.best_estimator_
scores = cross_val_score(best_clf, X_pca, y.ravel(), scoring='accuracy', cv=LeaveOneOut(), n_jobs=-1)
# report performance
print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

Accuracy: 0.867 (0.340)


In [ ]:
print(scores)

[0. 0. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1.]


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=5000, test_size=.1, random_state=0)
sss.get_n_splits(X_pca, y.ravel())
print(sss)

StratifiedShuffleSplit(n_splits=5000, random_state=0, test_size=0.1,
            train_size=None)


In [ ]:
best_clf1 = clf.best_estimator_
scores = cross_val_score(best_clf1, X_pca, y.ravel(), scoring='accuracy', cv=sss, n_jobs=-1)
# report performance
print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

Accuracy: 0.872 (0.181)


In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# Set the parameters by cross-validation
tuned_parameters = [{ 'max_features': [1, 2,4]}]

clf = GridSearchCV(RandomForestClassifier(), tuned_parameters, scoring='accuracy', cv=5)
clf.fit(X_pca, y.ravel())

print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
print("Grid scores on development set:")
print()
means = clf.cv_results_['mean_test_score']
stds = clf.cv_results_['std_test_score']
for mean, std, params in zip(means, stds, clf.cv_results_['params']):
        print("%0.3f (+/-%0.03f) for %r"
              % (mean, std * 2, params))

Best parameters set found on development set:

{'max_features': 4}

Grid scores on development set:

0.633 (+/-0.327) for {'max_features': 1}
0.633 (+/-0.249) for {'max_features': 2}
0.667 (+/-0.422) for {'max_features': 4}
